# Symbolic verification of a tenth-degree polynomial equation

This notebook is the computational companion to *A Tenth-Degree Polynomial Equation: Factorization, Roots of Unity, and Cyclotomic Polynomials*, by Osvaldo L. Santos-Pereira. Its purpose is to make the algebraic statements in the manuscript independently reproducible rather than merely to display numerical solutions. The module `verify_polynomial_10th_degree.py` implements exact polynomial expansions, factorizations, coefficient comparisons, symbolic simplifications, polynomial greatest common divisors, square-free decompositions, and finite parameter sweeps using SymPy. The notebook calls those functions in the logical order of the paper: starting with the exponent-block structure of the original tenth-degree equation, proceeding through its roots and their multiplicities, and finally testing the generalizations involving Chebyshev polynomials and products of finite geometric sums.

**How to interpret a result.** The message **VERIFIED | N/N exact checks satisfied** means that *all N mathematical assertions actually evaluated by that function were true*; it is not a synonym for “the program ran.” Every assertion uses an exact symbolic equality, an integer identity, a polynomial degree or root-count comparison, or an explicitly stated structural test. If an assertion is false, the function raises an `AssertionError` rather than printing **VERIFIED**. For a symbolic identity, equality is decided by cancellation of its algebraic difference; for a parameter sweep, the statement is established only on the specified finite grid. Neither finite sampling nor a successful computation is presented as a proof for all possible parameters. The published mathematical arguments supply those proofs; the code supplies independent, inspectable consistency checks.

**Reading convention.** Each Markdown cell states the result being tested, the algebraic reason for expecting it to hold, the domain or parameter range investigated, and the meaning of a successful verification. Its following code cell invokes the corresponding function in the module. The final cell reruns the complete verification suite and reports the cumulative number of exact assertions satisfied.


## Setup and notation

We work in the polynomial ring $\mathbb{Z}[x]$ whenever the relevant expressions have integer coefficients, extending to $\mathbb{Q}[x]$ or algebraic numbers when factorization into real quadratic factors or exact roots of unity requires it. The symbol $P$ denotes the specific monic polynomial of degree ten whose coefficient support is $\{0,1,2,4,5,6,8,9,10\}$. The finite geometric sum $S_m$ and the product families $P_{m,b}$ and $Q_{m,n,b}$ are constructed directly from this notation. SymPy represents polynomials symbolically, so the calculations avoid floating-point approximations and rounding-dependent comparisons.

$$
P(x)=1+x+x^2+x^4+x^5+x^6+x^8+x^9+x^{10}.
$$

**What the next cell does.** It imports SymPy and the verification module, exposes the central symbols $x,t,P,S,Q,P_{m,b},V_j$, records the installed SymPy version, and displays $P(x)$ in rendered LaTeX. This setup cell does not assert a mathematical result; it establishes that the computational environment is ready and that the polynomial being investigated is the intended one. Every subsequent status message is produced only after a dedicated verification function completes all its mathematical assertions.


In [1]:
import sympy as sp
from IPython.display import display, Math
import verify_polynomial_10th_degree as v
from verify_polynomial_10th_degree import x,t,P,S,Q,Pmb,V
print('SymPy version:',sp.__version__)
display(Math(r'P(x)='+sp.latex(P)))

SymPy version: 1.14.0


<IPython.core.display.Math object>

## 1. Exponent blocks

The missing coefficients of $x^3$ and $x^7$ are evidence of an exponent pattern rather than an obstacle to factorization. Every nonzero monomial has a uniquely determined exponent $r+4s$ with $r,s\in\{0,1,2\}$: the index $r$ selects a position inside a three-term block, while $s$ selects one of three blocks beginning at exponents $0,4,$ and $8$. Uniqueness matters because it ensures that expanding the product creates precisely nine monomials, each with coefficient one, without cancellation or repeated exponents. The factorization thus follows from separating two independent finite sums.

$$
P(x)=\sum_{s=0}^{2}\sum_{r=0}^{2}x^{r+4s}=(1+x+x^2)(1+x^4+x^8).
$$

**Verification and criterion.** The function `check_original_blocks()` compares the original polynomial with both the explicit block sum and the product $S_3(x)S_3(x^4)$ by exact symbolic subtraction and simplification. It separately checks that the nine generated exponents are distinct, equal the support written in the manuscript, and yield degree ten. **VERIFIED** means all five of these statements were satisfied, not simply that SymPy expanded an expression.


In [2]:
print(v.check_original_blocks())

VERIFIED | 5/5 exact checks satisfied | Exponent blocks: support, degree and factorization


## 2. Finite geometric sums

The factorization method rests on the finite geometric-series identity, which is established by multiplying the sum by $z-1$ and observing that intermediate powers cancel. This identity is polynomial and therefore remains valid at $z=1$; only the quotient formula $S_m(z)=(z^m-1)/(z-1)$ requires $z\neq1$. In particular, $S_m(1)=m$ follows from direct evaluation of the original sum. The distinction is essential when locating roots: $z=1$ is a zero of the numerator $z^m-1$ but never a zero of $S_m(z)$ for positive $m$.

$$
S_m(z)=\sum_{j=0}^{m-1}z^j,\qquad(z-1)S_m(z)=z^m-1,\qquad S_m(1)=m.
$$

**Verification and criterion.** The function `check_finite_geometric_sums()` tests the telescoping identity after the substitution $z=x^b$, with $2\le m\le12$ and $1\le b\le8$, and checks the special evaluation at $z=1$ for every parameter pair. Equality is established by exact symbolic cancellation. Its **176/176** result certifies every tested substitution and endpoint evaluation on this finite grid; the algebraic proof of the identity itself is the telescoping argument.


In [3]:
print(v.check_finite_geometric_sums())

VERIFIED | 176/176 exact checks satisfied | Finite geometric series: telescoping and z=1


## 3. Orders of roots of unity

A complex number is an $n$-th root of unity if its $n$-th power is one, but it need not have order exactly $n$. For the indexed roots $\omega_{n,h}=e^{2\pi ih/n}$, the least positive exponent returning the number to one is determined by the divisibility condition $n\mid hr$ and hence by the greatest common divisor of $n$ and $h$. Partitioning roots by their exact multiplicative orders produces the cyclotomic factorization of $x^n-1$ and, upon comparing degrees, the divisor identity for Euler's totient function. These are different consequences of the same cyclic structure.

$$
\omega_{n,h}=e^{2\pi ih/n},\quad \operatorname{ord}(\omega_{n,h})=\frac{n}{\gcd(n,h)}.
$$

$$
x^n-1=\prod_{d\mid n}\Phi_d(x),\qquad n=\sum_{d\mid n}\varphi(d).
$$

**Verification and criterion.** For each $1\le n\le12$, `check_roots_of_unity()` multiplies the cyclotomic factors over all positive divisors of $n$ and compares the product exactly with $x^n-1$. It verifies the totient divisor sum and tests the integer-divisibility and coprimality consequences of the index formula for every $0\le h<n$. The arithmetic checks use the known order formula; they are consistency tests and are not falsely described as independent complex-exponential proofs of minimality. **180/180 verified** records the exact outcome over the stated ranges.


In [4]:
print(v.check_roots_of_unity())

VERIFIED | 180/180 exact checks satisfied | Roots of unity and orders: order and divisor decomposition


## 4. Cyclotomic polynomials through order 12

The cyclotomic polynomial $\Phi_n$ contains precisely the primitive $n$-th roots of unity, each with multiplicity one. Its degree therefore equals $\varphi(n)$, and its factors combine according to $x^n-1=\prod_{d\mid n}\Phi_d(x)$. Computing the first twelve polynomials makes the abstract classification explicit. The twelfth-order case is particularly important because $\Phi_{12}(x)=x^4-x^2+1$ appears as an irreducible rational factor of the motivating tenth-degree polynomial, while $\Phi_3$ and $\Phi_6$ account for the lower-order roots.

$$
\Phi_n(x)=\prod_{\substack{0\le h<n\\\gcd(h,n)=1}}(x-\omega_{n,h}),\qquad \deg\Phi_n=\varphi(n).
$$

**Verification and criterion.** The function `check_cyclotomic_polynomials()` compares SymPy's exact cyclotomic polynomial against the explicit formula listed in the manuscript for every integer $1\le n\le12$. For each $n$, it also checks the degree against Euler's totient and invokes the exact irreducibility predicate over $\mathbb{Q}$. The output **36/36 verified** means that all twelve formulas, twelve degree equalities, and twelve irreducibility tests were satisfied. This checks the stated small-order cases; it does not replace the general theorem asserting irreducibility for arbitrary $n$.


In [5]:
print(v.check_cyclotomic_polynomials())

VERIFIED | 36/36 exact checks satisfied | Cyclotomic table Phi_1 through Phi_12: forms, degrees, irreducibility


## 5. Original cyclotomic factorization

The original factorization $P=S_3(x)S_3(x^4)$ resolves further into factors classified by exact multiplicative order. The first geometric sum equals $\Phi_3$, while the second equals $\Phi_3\Phi_6\Phi_{12}$, so the primitive cubic roots occur in both factors. This common occurrence accounts for their multiplicity two. A complementary algebraic diagnostic is $\gcd(P,P')$: in characteristic zero, a factor shared by a polynomial and its derivative detects its repeated roots. For this polynomial the gcd is the monic quadratic $\Phi_3$.

$$
P(x)=\Phi_3(x)^2\Phi_6(x)\Phi_{12}(x),\qquad\gcd(P,P')=\Phi_3(x).
$$

**Verification and criterion.** The function `check_original_cyclotomic_factorization()` establishes four exact statements: the complete cyclotomic product for $P$, the cyclotomic product for its degree-eight factor, the corresponding elementary factorization into explicit integer polynomials, and $\gcd(P,P')=\Phi_3$. The symbolic comparisons simplify polynomial differences to zero, while the greatest common divisor is computed in $\mathbb{Q}[x]$. **VERIFIED** therefore certifies both the factorization and the location of repeated algebraic factors.


In [6]:
print(v.check_original_cyclotomic_factorization())

VERIFIED | 4/4 exact checks satisfied | Original cyclotomic factorization: Phi_3^2 Phi_6 Phi_12, gcd(P,P')


## 6. Multiplicity balance at the twelve candidate roots

The polynomial identity obtained from the geometric sums gives more information than the mere fact that the roots are twelfth roots of unity. At each candidate $\omega_{12,h}$, the multiplicity of a product is the sum of the multiplicities of its factors. Both right-hand binomials in the identity have simple zeros; the factors multiplying $P$ on the left remove the four axial points. An indicator-function formula then distinguishes the two primitive cubic roots, where the multiplicity is two, from the six remaining non-axial roots, where it is one. The sum of these multiplicities must equal ten.

$$
(1-x)(1-x^4)P(x)=(1-x^3)(1-x^{12}),
$$

$$
\operatorname{mult}_{P}(\omega_{12,h})=1+\mathbf1_{4\mid h}-\mathbf1_{3\mid h}-\mathbf1_{h=0}.
$$

**Verification and criterion.** The function `check_multiplicity_balance()` first verifies the full polynomial identity by exact expansion. It then evaluates the indicator formula at each of the twelve indices $h=0,\dots,11$, checks that the square-free decomposition of $P$ consists of eight distinct roots (six simple and two double), verifies total multiplicity ten, and confirms that the gcd of $P$ with $x^{12}-1$ has degree eight. **16/16 verified** means every classified candidate point and the independent algebraic root counts agree.


In [7]:
print(v.check_multiplicity_balance())

VERIFIED | 16/16 exact checks satisfied | Twelve candidate roots and multiplicities: four excluded, six simple, two double


## 7. Independent substitution

A second derivation treats the degree-eight factor $1+x^4+x^8$ as a quadratic in $u=x^4$. The two nonzero solutions $u=e^{\pm2\pi i/3}$ each have four distinct fourth roots, giving eight solutions of the second factor. Writing these solutions as angular progressions is useful because it makes comparison with the roots of the other factor immediate. The primitive cubic roots $\omega$ and $\omega^2$ belong to both zero sets: they satisfy $x^4=x$ and hence also the substituted quadratic. These are precisely the two repeated roots of $P$.

$$
1+u+u^2=0,\quad u=e^{\pm 2\pi i/3},
$$

$$
\theta=\pm\frac{\pi}{6}+\frac{k\pi}{2},\qquad k=0,1,2,3.
$$

**Verification and criterion.** The function `check_fourth_power_substitution()` verifies the substitution as an exact polynomial identity, enumerates the four residue classes for each angular sign, confirms that the resulting index sets are $\{1,4,7,10\}$ and $\{2,5,8,11\}$ without overlap, and identifies indices $4$ and $8$ as shared with the first factor. **5/5 verified** certifies these algebraic and combinatorial conditions; the angular classification uses exact integer residues rather than approximate trigonometric evaluation.


In [8]:
print(v.check_fourth_power_substitution())

VERIFIED | 5/5 exact checks satisfied | Substitution u=x^4: eight fourth roots and two shared roots


## 8. Real quadratic factors

Complex conjugation pairs the nonreal roots of a real polynomial. Each pair $e^{i\theta}$ and $e^{-i\theta}$ produces the real quadratic $x^2-2x\cos\theta+1$, and the four relevant angular classes yield the complete real factorization of $P$. The two factors with coefficients $\pm\sqrt3$ combine into the integer-coefficient quartic $\Phi_{12}$, explaining the relationship between factorizations over $\mathbb{R}$ and $\mathbb{Q}$. None of the real quadratic factors has a real zero because its discriminant is strictly negative for each of the four relevant cosine values.

$$
P(x)=(x^2+x+1)^2(x^2-x+1)(x^2-\sqrt3x+1)(x^2+\sqrt3x+1).
$$

**Verification and criterion.** The function `check_real_factorization()` verifies the product of the conjugate irrational quadratics, verifies the complete tenth-degree factorization, and tests the strict negativity of the four discriminants $t_0^2-4$ with $t_0\in\{-1,1,\sqrt3,-\sqrt3\}$. SymPy evaluates these signs exactly with radicals, without numerical tolerance. **6/6 verified** therefore establishes the displayed factorization and the absence of real zeros through this quadratic decomposition.


In [9]:
print(v.check_real_factorization())

VERIFIED | 6/6 exact checks satisfied | Real quadratic factorization: all real quadratics have negative discriminant


## 9. Viète and symmetric sums

Viète's formulas provide coefficient-based checks that do not depend on the manner in which the polynomial was factored. Because the polynomial has degree ten and contains two double roots, the sequence of ten roots must repeat the primitive cubic roots before symmetric sums are formed. The coefficient of $x^9$ determines the first elementary symmetric sum, the coefficient of $x^8$ determines the second, and the constant coefficient determines the product of the roots. The identity $2e_2=(\sum r_j)^2-\sum r_j^2$ connects the second elementary symmetric sum with the second power sum.

$$
e_1=\sum r_j=-1,\qquad e_2=\frac{(\sum r_j)^2-\sum r_j^2}{2}=1,\qquad e_{10}=\prod r_j=1.
$$

**Verification and criterion.** The function `check_viete()` tests the four relevant polynomial coefficients, constructs the exact list of ten roots (including repeated indices $4$ and $8$), and evaluates their sum, squared-power sum, product and induced value of $e_2$ using exact algebraic simplification of the complex exponentials. **5/5 verified** means these calculations reproduce $e_1=-1$, $\sum r_j^2=-1$, $e_2=1$, and $e_{10}=1$, consistently with the corresponding coefficients of $P$.


In [10]:
print(v.check_viete())

VERIFIED | 5/5 exact checks satisfied | Viete and power sums: e1=-1, e2=1, e10=1


## 10. Chebyshev recurrence and symmetric powers

The substitution $t=x+x^{-1}$ replaces inversion-symmetric Laurent monomials by ordinary polynomials. This follows from the recurrence satisfied by $x^j+x^{-j}$, which is identical to the recurrence for $2T_j(t/2)$. Chebyshev polynomials of the second kind satisfy a companion recurrence, providing an independent check on the standard definitions and initial conditions. Establishing these identities for representative indices justifies the explicit polynomials $V_0,\ldots,V_5$ used in the paper and helps prevent indexing or normalization errors such as missing factors of two.

$$
T_{j+1}(y)=2yT_j(y)-T_{j-1}(y),\qquad U_{j+1}(y)=2yU_j(y)-U_{j-1}(y),
$$

$$
V_j(t)=2T_j(t/2),\qquad V_j(x+x^{-1})=x^j+x^{-j}.
$$

**Verification and criterion.** The function `check_chebyshev_recurrences()` tests both standard recurrences for indices $1\le j\le12$, then checks the identities $V_j(t)=2T_j(t/2)$ and $V_j(x+x^{-1})=x^j+x^{-j}$ for $0\le j\le10$. The latter comparisons are exact rational-function identities for nonzero $x$. **46/46 verified** certifies these finite-index instances; the recurrence proof in the manuscript establishes the general result for arbitrary nonnegative $j$.


In [11]:
print(v.check_chebyshev_recurrences())

VERIFIED | 46/46 exact checks satisfied | Chebyshev and symmetric powers: T, U, V recurrences and Laurent identities


## 11. Reciprocal reduction of the tenth-degree polynomial

A polynomial $F$ of even degree $2m$ is reciprocal when its coefficients are symmetric and $F(x)=x^{2m}F(x^{-1})$ for nonzero $x$. For the tenth-degree polynomial, dividing by $x^5$ produces the inversion-invariant combination $x^5+x^{-5}+x^4+x^{-4}+x^3+x^{-3}+x+x^{-1}+1$. The auxiliary polynomials $V_j$ translate this expression into a quintic in $t=x+x^{-1}$. Its factorization into $(t+1)^2(t-1)(t^2-3)$ recovers the same four real quadratic factors after lifting each linear factor to $x^2-t_0x+1$.

$$
x^{-5}P(x)=t^5+t^4-4t^3-4t^2+3t+3=(t+1)^2(t-1)(t^2-3).
$$

**Verification and criterion.** The function `check_reciprocal_reduction()` compares the sum of the auxiliary polynomials with the expanded quintic, checks its factorization, substitutes $t=x+x^{-1}$ back into both equivalent quintic representations and verifies that multiplication by $x^5$ reconstructs $P$. It also checks symmetry of the original coefficient vector. **5/5 verified** ensures that the change of variables, factorization, inverse substitution and reciprocal condition agree algebraically.


In [12]:
print(v.check_reciprocal_reduction())

VERIFIED | 5/5 exact checks satisfied | Reciprocal quintic reduction: fifth-degree reduction and its factorization


## 12. General reciprocal-polynomial theorem

The preceding calculation is an instance of a general reduction: every reciprocal polynomial of exact degree $2m$ can be uniquely expressed as $x^mR(x+x^{-1})$ with $\deg R=m$. The reason is that the coefficients of $x^j$ and $x^{2m-j}$ agree; after factoring out $x^m$ these terms become scalar multiples of symmetric powers $x^k+x^{-k}$, each represented by $V_k(t)$. The midpoint coefficient contributes a constant term. This is an algebraic consequence of the recurrence and applies to nonzero values of $x$.

$$
F(x)=x^{2m}F(x^{-1})\ \Longrightarrow\ F(x)=x^mR(x+x^{-1}),\qquad \deg R=m.
$$

**Verification and criterion.** The function `check_general_reciprocal_reduction()` constructs representative reciprocal polynomials with explicitly chosen symmetric coefficients for each $1\le m\le9$, assembles the associated $R$ from the $V_j$ basis, verifies $F(x)=x^mR(x+x^{-1})$ by exact cancellation, and checks that $\deg R=m$. **18/18 verified** demonstrates these examples. Since it tests one coefficient pattern per degree, it is a regression test of the implementation, not a universal computational proof for arbitrary coefficients.


In [13]:
print(v.check_general_reciprocal_reduction())

VERIFIED | 18/18 exact checks satisfied | General reciprocal reduction: degree 2m to degree m


## 13. Separated blocks in arbitrary geometric products

The observation that produced the original factorization generalizes to products of sums with $m$ consecutive monomials and $q$ exponent blocks separated by a step $b$. Under the condition $b\ge m$, the map $(r,s)\mapsto r+bs$ is injective for $0\le r<m$ and $0\le s<q$; otherwise two distinct selections could generate the same exponent and increase a coefficient above one. Injectivity therefore guarantees that the expanded product has exactly $mq$ nonzero coefficients, all equal to one, and degree $(m-1)+b(q-1)$.

$$
\sum_{s=0}^{q-1}\sum_{r=0}^{m-1}x^{r+bs}=S_m(x)S_q(x^b),\quad \deg=m-1+b(q-1).
$$

**Verification and criterion.** The function `check_separated_exponent_blocks()` runs over $2\le m\le7$, $2\le q\le6$, and the five admissible steps $m\le b\le m+4$. For every triple it counts distinct exponents, compares the double sum against $S_m(x)S_q(x^b)$, and checks the predicted degree. **450/450 verified** records the exact agreement across this finite collection of separated-block configurations, including both adjacent and gapped blocks.


In [14]:
print(v.check_separated_exponent_blocks())

VERIFIED | 450/450 exact checks satisfied | Separated-block criterion: support, factorization and degree


## 14. Products of equal-length sums

Consider $P_{m,b}(x)=S_m(x)S_m(x^b)$. The first factor has zero set $\mathcal U_m\setminus\{1\}$, and the second has zero set $\mathcal U_{mb}\setminus\mathcal U_b$. Their common zeros form $\mathcal U_m\setminus\mathcal U_b$, whose cardinality is $m-\gcd(m,b)$. Since each individual geometric sum has simple zeros, these common points become double zeros of the product and all other zeros remain simple. Inclusion–exclusion then gives $b(m-1)+\gcd(m,b)-1$ distinct roots. This explains how arithmetic divisibility governs the multiplicity distribution of a potentially high-degree polynomial.

$$
N_{\rm double}=m-g,\quad N_{\rm distinct}=b(m-1)+g-1,\quad\deg P_{m,b}=(m-1)(b+1).
$$

**Verification and criterion.** The function `check_equal_length_counts()` tests the complete root-count statement for $2\le m\le8$ and $1\le b\le8$. Rather than approximating complex roots numerically, it uses SymPy's exact square-free factorization to count roots of multiplicity one and two, checks the predicted degree and distinct-root count, verifies the degree identity $\text{simple}+2\text{double}=\deg P_{m,b}$, and computes the polynomial gcd of the factors to measure overlap. **280/280 verified** means these independent algebraic counts agree for all tested pairs.


In [15]:
print(v.check_equal_length_counts())

VERIFIED | 280/280 exact checks satisfied | Equal-length sums P_{m,b}: gcd overlap and square-free root counts


## 15. Products of unequal-length sums

When the two geometric sums have different lengths, the common roots must satisfy two divisibility constraints simultaneously. For $Q_{m,n,b}(x)=S_m(x)S_n(x^b)$, the first factor contributes $\mathcal U_m\setminus\{1\}$, while the second contributes $\mathcal U_{nb}\setminus\mathcal U_b$. Their intersection is $\mathcal U_{\gcd(m,nb)}\setminus\mathcal U_{\gcd(m,b)}$, giving $\delta_{m,n,b}=\gcd(m,nb)-\gcd(m,b)$ double roots. Since both factors have only simple zeros, the number of distinct roots is their total degree minus this overlap. This extends the equal-length counting result while preserving its arithmetic interpretation.

$$
\delta_{m,n,b}=\gcd(m,nb)-\gcd(m,b),
$$

$$
\nu_{m,n,b}=(m-1)+b(n-1)-\delta_{m,n,b}.
$$

**Verification and criterion.** The function `check_unequal_length_counts()` uses every triple $2\le m,n\le7$, $1\le b\le6$, giving 216 parameter combinations. For each triple it checks the degree, the predicted double-root count, the distinct-root count, the weighted multiplicity total, and the degree of the gcd of the two factors against an exact square-free decomposition. **1080/1080 verified** means all five conditions agree for each of the 216 instances; it does not claim that a finite grid proves the formula for all integers.


In [16]:
print(v.check_unequal_length_counts())

VERIFIED | 1080/1080 exact checks satisfied | Unequal-length sums Q_{m,n,b}: gcd difference and root counts


## 16. Real-root parity criterion

All zeros of the geometric-sum factors are roots of unity, so a real root must be either $1$ or $-1$. Because $S_m(1)=m$ and $S_n(1)=n$, the product never vanishes at $1$. At $-1$, the first factor vanishes exactly when $m$ is even. The second factor vanishes exactly when $n$ is even and the dilation exponent $b$ is odd, because an even $b$ sends $-1$ to $1$. Each zero of either individual factor is simple, so the multiplicity of $-1$ in the product is the sum of two indicator functions and can only be zero, one or two.

$$
\operatorname{mult}_{Q_{m,n,b}}(-1)=\mathbf1_{2\mid m}+\mathbf1_{2\mid n,\ 2\nmid b}.
$$

**Verification and criterion.** The function `check_real_root_parity()` ranges over $2\le m,n\le8$ and $1\le b\le6$. It checks that the value of the product at $-1$ is zero exactly when the indicator sum is positive; when a zero is predicted, it tests the appropriate vanishing derivatives and the first nonvanishing derivative. In the non-root case it also checks the positive evaluations at $x=0$ and $x=1$. **882/882 verified** establishes the finite-grid parity classification, including multiplicity, without estimating roots numerically.


In [17]:
print(v.check_real_root_parity())

VERIFIED | 882/882 exact checks satisfied | Real-root parity criterion: multiplicity of -1 is zero, one or two


## 17. The article's worked exercises

The final examples apply the general identities to concrete polynomials selected to illustrate different types of overlap between geometric sums. The degree-six product $S_3(x)S_3(x^2)$ contains two double primitive cube roots; $P_{6,4}$ has degree 25 and four double roots; and $Q_{6,4,3}$ contains the double real root $-1$ together with two double primitive sixth roots. Additional examples include $P_{4,2}$, whose roots $i$ and $-i$ are double, and $P_{3,3}=S_9$, which is square-free. These cases test whether the general counting routines reproduce the examples actually stated in the manuscript.

$$
S_3(x)S_3(x^2)=\Phi_3(x)^2\Phi_6(x),
$$

$$
Q_{6,4,3}(x)=S_6(x)S_4(x^3),\quad N_{\rm distinct}=11,\quad N_{\rm double}=3.
$$

**Verification and criterion.** The function `check_worked_examples()` computes seven exact square-free root-count quadruples consisting of the numbers of distinct, simple and double roots together with the degree. It additionally verifies four explicit identities: the degree-six cyclotomic decomposition, the factorization of $P_{4,2}$, the equality $P_{3,3}=S_9$, and the gcd identifying the common zeros of $Q_{6,4,3}$. **11/11 verified** means every published example matched its prescribed symbolic factorization or root count.


In [18]:
print(v.check_worked_examples())

VERIFIED | 11/11 exact checks satisfied | Paper examples and four exercises: seven examples and cyclotomic checks


## 18. Full regression

A reproducible verification should test both the isolated identities and their compatibility in one uninterrupted execution. The final code cell invokes every one of the seventeen verification functions again, independently of the results printed by the earlier cells, and checks that the returned result collection has the expected length. The aggregate count is the sum of the explicitly counted conditions inside those functions; it is not the number of Python statements executed, and it must not be interpreted as a measure of the strength of a mathematical proof. The primary original-polynomial conclusion remains that $P$ has eight distinct roots, six simple and two double, with ten roots counted with multiplicity.

$$
\sum_{\alpha:P(\alpha)=0}\operatorname{mult}_P(\alpha)=\deg P=10.
$$

**Interpretation of the final report.** The line **ALL VERIFICATIONS SATISFIED | 3205 exact checks | 0 failures** is printed only if all seventeen routines return normally, without a failed assertion. It means that every exact symbolic comparison and every finite-grid condition implemented by the module was satisfied in that execution. If any identity, multiplicity count, irreducibility test, parity criterion or example disagrees with its expected result, an exception prevents this success report. Formal proofs in the manuscript remain necessary for the parameterized theorems.


In [19]:
results=v.run_all(verbose=True)
assert len(results)==len(v.CHECKS)

VERIFIED | 5/5 exact checks satisfied | Exponent blocks: support, degree and factorization
VERIFIED | 176/176 exact checks satisfied | Finite geometric series: telescoping and z=1
VERIFIED | 180/180 exact checks satisfied | Roots of unity and orders: order and divisor decomposition
VERIFIED | 36/36 exact checks satisfied | Cyclotomic table Phi_1 through Phi_12: forms, degrees, irreducibility
VERIFIED | 4/4 exact checks satisfied | Original cyclotomic factorization: Phi_3^2 Phi_6 Phi_12, gcd(P,P')
VERIFIED | 16/16 exact checks satisfied | Twelve candidate roots and multiplicities: four excluded, six simple, two double
VERIFIED | 5/5 exact checks satisfied | Substitution u=x^4: eight fourth roots and two shared roots
VERIFIED | 6/6 exact checks satisfied | Real quadratic factorization: all real quadratics have negative discriminant
VERIFIED | 5/5 exact checks satisfied | Viete and power sums: e1=-1, e2=1, e10=1
VERIFIED | 46/46 exact checks satisfied | Chebyshev and symmetric powers: T, 